[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gauravs19/iiot-predictive-maintenance/blob/main/notebooks/03_anomaly_detection.ipynb)

# 03 · Anomaly Detection (unsupervised)

In the real world you usually **don't have labelled failures** — failures are rare,
and labelling every sensor reading is impractical. So the realistic question becomes:

> *Given mostly-normal operating data, can we automatically flag readings that look
> abnormal — potential early warnings — without ever being told what a failure looks
> like?*

This is **unsupervised anomaly detection**. We use two complementary techniques and
compare them:

1. **Isolation Forest** — a tree-based method that isolates outliers.
2. **Autoencoder** — a neural net trained to reconstruct *healthy* data; anything it
   reconstructs poorly is anomalous.

We then validate the unsupervised scores against the RUL we *do* have, to show the
anomaly score genuinely rises as engines approach failure.

## Step 1 · Bootstrap + imports

In [ ]:
# --- Environment bootstrap (works locally AND on Google Colab) ---------------
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # On Colab there is no repo yet, so clone it and install dependencies.
    !git clone -q https://github.com/gauravs19/iiot-predictive-maintenance.git
    %cd iiot-predictive-maintenance
    !pip install -q -r requirements.txt

# Make the repo root importable so `from src import ...` works from notebooks/.
def _find_repo_root(start="."):
    p = os.path.abspath(start)
    while p != os.path.dirname(p):
        if os.path.isdir(os.path.join(p, "src")):
            return p
        p = os.path.dirname(p)
    raise RuntimeError("repo root (folder containing src/) not found")

REPO_ROOT = _find_repo_root()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)
print("running on Colab" if IN_COLAB else "running locally")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

from src import data, features, models, utils
print("imports OK")

## Step 2 · Define "healthy" vs "degraded"

We have no failure labels, but we *do* know each training row's RUL. We use that only
to **construct an evaluation set** (not to train the detectors):

- **Healthy** = rows with high RUL (engine early in life).
- **Degraded** = rows with low RUL (engine near failure) — these *should* score as
  anomalous if our detectors work.

The detectors themselves are trained **only on healthy data**, mimicking a real
deployment where you fit on normal operation and watch for deviations.

In [ ]:
cm = data.load_cmapss("FD001")
df = features.add_rolling_features(features.add_rul(cm["train"], clip=125),
                                   features.feature_columns(cm["train"]))
cols = features.feature_columns(cm["train"])

healthy = df[df["rul"] >= 100]   # train detectors on these
degraded = df[df["rul"] <= 20]   # should be flagged as anomalous
print("healthy rows:", len(healthy), " degraded rows:", len(degraded))

scaler = utils.Standardizer().fit(healthy[cols].to_numpy("float32"))
Xh = scaler.transform(healthy[cols].to_numpy("float32"))
Xd = scaler.transform(degraded[cols].to_numpy("float32"))

## Step 3 · Method 1 — Isolation Forest

**Intuition:** an Isolation Forest builds random trees that repeatedly split the data
on random features. Outliers are "easy to isolate" — they get separated from the herd
in only a few splits, so they end up with **short path lengths**. The model turns that
into an anomaly score.

We fit it on healthy data only, then score both healthy and degraded rows. We expect
the degraded rows to score as anomalies far more often.

In [ ]:
iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
iso.fit(Xh)

# decision_function: higher = more normal. We negate so higher = more anomalous.
score_h = -iso.decision_function(Xh)
score_d = -iso.decision_function(Xd)
print("mean anomaly score — healthy: %.3f   degraded: %.3f" % (score_h.mean(), score_d.mean()))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(score_h, bins=40, alpha=0.6, label="healthy", density=True)
ax.hist(score_d, bins=40, alpha=0.6, label="degraded", density=True)
ax.set_title("Isolation Forest anomaly scores"); ax.legend(); plt.show()

## Step 4 · Method 2 — Autoencoder (reconstruction error)

**Intuition:** an **autoencoder** is a neural net that compresses its input to a small
"bottleneck" and then reconstructs it. If we train it *only on healthy data*, it
becomes expert at rebuilding normal patterns — but when shown a degraded reading it
has never learned, it reconstructs it **poorly**. That **reconstruction error** is our
anomaly score.

This is the same encoder idea that will later feed the `iiot-ai-rag` project: the
bottleneck layer is effectively a learned "signature" of a machine's state.

In [ ]:
ae = models.AutoEncoder(n_features=Xh.shape[1], latent=8)
hist = models.train_autoencoder(ae, Xh, epochs=30, batch_size=256)
utils.plot_loss(hist, "Autoencoder reconstruction loss (healthy data)"); plt.show()

In [ ]:
err_h = models.reconstruction_error(ae, Xh)
err_d = models.reconstruction_error(ae, Xd)
print("mean reconstruction error — healthy: %.4f   degraded: %.4f"
      % (err_h.mean(), err_d.mean()))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(err_h, bins=40, alpha=0.6, label="healthy", density=True)
ax.hist(err_d, bins=40, alpha=0.6, label="degraded", density=True)
ax.set_title("Autoencoder reconstruction error"); ax.legend(); plt.show()

## Step 5 · Quantify: how well do the scores separate the two groups?

The histograms look convincing, but let's be rigorous. We treat "degraded" as the
positive class and compute **ROC-AUC** of each anomaly score — i.e. if we pick a
random degraded row and a random healthy row, how often does the detector score the
degraded one higher? **1.0 = perfect separation, 0.5 = useless.**

This is a fair check *because* the detectors never saw the RUL labels — we're using
them only to grade the unsupervised result.

In [ ]:
y = np.r_[np.zeros(len(Xh)), np.ones(len(Xd))]   # 0=healthy, 1=degraded
auc_iso = roc_auc_score(y, np.r_[score_h, score_d])
auc_ae  = roc_auc_score(y, np.r_[err_h, err_d])
print("ROC-AUC  Isolation Forest: %.3f" % auc_iso)
print("ROC-AUC  Autoencoder     : %.3f" % auc_ae)

## Step 6 · Does the anomaly score rise as a real engine ages?

The ultimate test of usefulness: track one engine across its whole life and plot its
anomaly score over time. A good detector's score should stay low while the engine is
healthy and **climb steadily as it nears failure** — turning into an actionable early
warning.

In [ ]:
unit = df[df["unit"] == 1].sort_values("cycle")
Xu = scaler.transform(unit[cols].to_numpy("float32"))
err_u = models.reconstruction_error(ae, Xu)

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(unit["cycle"], err_u, color="crimson", label="anomaly score")
ax1.set_xlabel("cycle"); ax1.set_ylabel("reconstruction error", color="crimson")
ax2 = ax1.twinx()
ax2.plot(unit["cycle"], unit["rul"], color="steelblue", alpha=0.6, label="RUL")
ax2.set_ylabel("RUL (true)", color="steelblue")
ax1.set_title("Engine #1 — anomaly score climbs as RUL falls")
plt.show()

## Step 7 · Setting an alert threshold

To deploy this you turn the continuous score into a yes/no alarm by choosing a
**threshold**. A common, label-free choice is a high percentile of the *healthy*
score distribution (e.g. the 99th percentile) — meaning "alarm when the machine looks
more abnormal than 99% of its normal operation." Tightening or loosening this trades
**false alarms** against **missed detections**.

In [ ]:
thr = np.percentile(err_h, 99)
flagged = (err_d > thr).mean()
print("threshold (99th pct of healthy): %.4f" % thr)
print("share of degraded rows correctly flagged: %.1f%%" % (100 * flagged))
print("false-alarm rate on healthy rows         : %.1f%%" % (100 * (err_h > thr).mean()))

## Step 8 · Wrap-up & where this goes next

**What we built across the project:**

| Notebook | Capability | Technique |
|---|---|---|
| 00 | Data ingestion | UCI + NASA loaders |
| 01 | Feature engineering | RUL labels, rolling stats, sequences |
| 02 | Predictive maintenance | Random Forest + SHAP, LSTM RUL |
| 03 | Anomaly detection | Isolation Forest + Autoencoder |

**The hand-off to `iiot-ai-rag` (the sibling project):** the autoencoder's bottleneck
gives a compact **vector signature** of each machine state, and notebook `02`'s model
produces failure probabilities / RUL. Those outputs are exactly what a
Retrieval-Augmented-Generation layer will consume — embedding signatures into a vector
DB to retrieve similar past incidents, and letting an LLM write the maintenance
work-order. That's deliberately kept as a **separate project** so the ML core stays
clean and self-contained.